In [1]:
#%pip install torch
import torch
import torch.nn as nn
import math

In [2]:
class InputEmbedding(nn.Module):
    def __init__(self,d_model,vocab_size):
        super(InputEmbedding,self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size,d_model)
    def forward(self,x):
        return self.embedding(x) * math.sqrt(self.d_model)
    

In [3]:
d_model = 8
vocab_size = 1000
embed=InputEmbedding(d_model,vocab_size)
sentence_taken = torch.tensor([[10,25,500,30,31,85]])
output = embed(sentence_taken)
print(output.shape)

torch.Size([1, 6, 8])


In [4]:
print(output)

tensor([[[ 3.1661, -2.0497, -0.7327,  0.3106,  3.0193, -0.1794, -2.0129,
           4.4671],
         [ 2.3732, -2.7215,  4.6831,  2.7037, -0.5816, -4.9650,  4.2603,
           0.0067],
         [ 2.7518, -0.8504,  2.1631, -0.5415,  3.2146,  0.6255,  3.5749,
           3.7040],
         [-6.0558,  1.9429, -2.6347, -1.6356,  2.9183, -0.6533, -2.9518,
          -0.7311],
         [-0.6079, -4.3493, -1.9266,  5.2478,  1.2415, -2.9972,  1.5449,
           3.2067],
         [-1.0219,  2.2399, -1.9788,  1.1004, -2.5494, -3.1306, -2.2495,
           1.8297]]], grad_fn=<MulBackward0>)


In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model,squ_len: int,dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        self.squ_len = squ_len
        pe = torch.zeros(squ_len,d_model)
        position = torch.arange(0,squ_len,dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model,2).float() * (-math.log(10000.0) / d_model))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0,1)
        self.register_buffer('pe',pe)
        
    def forward(self,x):
        x = x + self.pe[:, :x.size(1), :].requires_grad_(False)
        return self.dropout(x)
        

In [6]:
d_model = 8
squ_len = 1000
pos_enc = PositionalEncoding(d_model,squ_len,dropout=0.1)
x=embed(sentence_taken)
output = pos_enc(x)
print(output.shape)
print(output)


torch.Size([1000, 6, 8])
tensor([[[ 3.5178, -1.1663, -0.8141,  ...,  0.9118, -2.2366,  6.0746],
         [ 2.6369, -1.9127,  5.2034,  ..., -4.4056,  4.7336,  1.1185],
         [ 3.0576,  0.1663,  2.4035,  ...,  1.8061,  3.9721,  5.2267],
         [-6.7286,  3.2699, -2.9275,  ...,  0.3852, -3.2798,  0.2987],
         [-0.6755, -3.7214, -2.1406,  ..., -2.2191,  1.7165,  4.6741],
         [-1.1355,  3.5999, -2.1986,  ..., -2.3673, -2.4995,  3.1441]],

        [[ 4.4528, -1.6771, -0.7032,  ...,  0.9118, -2.2355,  0.0000],
         [ 3.5719, -2.4235,  5.3144,  ..., -4.4056,  0.0000,  1.1185],
         [ 3.9926, -0.3445,  2.5144,  ...,  0.0000,  3.9732,  5.2267],
         [-5.7937,  2.7592, -2.8166,  ...,  0.3852, -3.2786,  0.2987],
         [ 0.2595, -0.0000, -0.0000,  ..., -2.2192,  1.7176,  4.6741],
         [-0.2005,  3.0891, -2.0877,  ..., -2.3674, -2.4984,  3.1441]],

        [[ 0.0000, -2.7398, -0.5934,  ...,  0.9116, -2.2343,  6.0746],
         [ 3.6473, -3.4862,  5.4242,  ..., -4.40

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()
        self.h = h
        self.d_model = d_model
        assert d_model % h == 0, "d_model must be divisible by h"
        
        self.head_dim = d_model // h
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.size(-1)
        # Scaled Dot-Product Attention
        # (Batch, h, seq_q, d_k) x (Batch, h, d_k, seq_k) -> (Batch, h, seq_q, seq_k)
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        
        attention_probs = F.softmax(attention_scores, dim=-1)
        
        if dropout is not None:
            attention_probs = dropout(attention_probs)
            
        # Multiply probs by value: (Batch, h, seq_q, seq_k) x (Batch, h, seq_k, head_dim)
        return torch.matmul(attention_probs, value), attention_probs

    def forward(self, q, k, v, mask):
        # Linear projections
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        # Split into h heads: (Batch, Seq, d_model) -> (Batch, Seq, h, head_dim) -> (Batch, h, Seq, head_dim)
        query = query.view(query.size(0), -1, self.h, self.head_dim).transpose(1, 2)
        key = key.view(key.size(0), -1, self.h, self.head_dim).transpose(1, 2)
        value = value.view(value.size(0), -1, self.h, self.head_dim).transpose(1, 2)

        x, self_attention_probs = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # Concatenate heads: (Batch, h, Seq, head_dim) -> (Batch, Seq, h, head_dim) -> (Batch, Seq, d_model)
        x = x.transpose(1, 2).contiguous().view(x.size(0), -1, self.d_model)

        return self.w_o(x)

In [9]:
class layerNormalization(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super(layerNormalization,self).__init__()
        self.d_model = d_model
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))
    def forward(self,x):
        mean = x.mean(dim=-1,keepdim=True)
        std = x.std(dim=-1,keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias
    

In [10]:
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))